In [0]:
%sql
CREATE DATABASE IF NOT EXISTS workspace.gold
COMMENT 'Capa Gold procesadoss'


In [0]:
%sql
DROP DATABASE IF EXISTS workspace.gold CASCADE; 

In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.gold.dim_ubicacion_proyecto")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.dim_ubicacion (
  ubicacion_id INT,
  divipola INT,
  departamento_descripcion STRING,
  municipio_descripcion STRING
)
""")

In [0]:
import pandas as pd
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window


# 1. Leer desde Silver directamente como Spark DataFrame
df_silver = spark.table("workspace.silver.predios_registro")


# 2. Seleccionar columnas necesarias y limpiar nulos 
dim_ubicacion = df_silver.select(
    "divipola",
    "departamento_descripcion",
    "municipio_descripcion"
)


dim_ubicacion = dim_ubicacion.fillna({
    "departamento_descripcion": "SIN INFORMACION",
    "municipio_descripcion": "SIN INFORMACION"
})

dim_ubicacion = dim_ubicacion.dropDuplicates(["divipola"])


# 3. Crear ID (clave sustituta)
window = Window.orderBy("divipola")
dim_ubicacion = dim_ubicacion.withColumn("ubicacion_id", row_number().over(window))

# 4. Reordenar columnas si quieres
dim_ubication = dim_ubicacion.select(
    "ubicacion_id",
    "divipola",
    "departamento_descripcion",
    "municipio_descripcion"
)


In [0]:

# Usamos overwrite para reemplazar completamente los datos existentes
dim_ubicacion.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.dim_ubicacion")